# LangChain · Meridian's Claims Assistant

Meridian Travel Insurance gets the same questions all day. *Am I covered if…? How much will I get back? Where's my claim?* Every answer depends on two things: the **policy wording**, which says what is covered at all, and the **customer's account** — their plan, their add-ons, the conditions they declared, the claims they have already made.

This is the kind of application LangChain is for: not an agent deciding what to do, but a fixed sequence of model calls, retrieval and lookups, composed into one thing that can be invoked, batched, streamed, retried and observed. Every message goes through the same four steps:

```
message ─▶ rewrite ─▶ ┌ retrieve: the clauses it is about ──┐ ─▶ respond ─▶ check
  + history           └ look up: the account facts ─────────┘    (a Reply)   (code)
```

| Step | What it does | LangChain pieces |
|---|---|---|
| **rewrite** | a follow-up — *"how long do I have to send it?"* — becomes a standalone question | `ChatPromptTemplate`, `MessagesPlaceholder`, `RunnableBranch`, `StrOutputParser` |
| **retrieve** | the policy clauses the question is about | `Embeddings`, `Document`, `InMemoryVectorStore`, a retriever |
| **look up** | the policy schedule and claims always; a claim's status or a payout estimate when the question needs one | structured output to extract, tools as Runnables |
| **respond** | clauses + facts → a `Reply`: covered or not, the payout, the clauses it rests on | `with_structured_output`, `with_fallbacks` |
| **check** | every clause cited must be one the model was shown | a `RunnableLambda` — plain code |

Built twice: **with LangChain** (`claims_chain.py`) and **from this repo's parts** (`claims_scratch.py`: chapter 1's `extract()` and `chat()`, chapter 3's tools, chapter 4's embeddings, chapter 1's `Conversation`).

Eight messages from three policyholders, in order — two of them are follow-ups, and four are traps:

| | Who | Message | The right answer |
|---|---|---|---|
| 1 | Priya, Plus | laptop stolen from her hotel room, locked in the safe, $1,800 | covered — a safe is not "unattended"; single-item limit $500 less $100 excess = **$400** |
| 2 | Priya | where's claim CLM-20931? | on hold, waiting for the police report requested 29 September |
| 3 | Priya | *"how long do I have to send it?"* | 30 days from the request: **29 October** — needs message 2 to make sense |
| 4 | Daniel, Essential | if I break my leg skiing in Austria, am I covered? | **no** — winter sports need an add-on he doesn't have |
| 5 | Daniel | my mum's in hospital, I might cancel | covered — a parent is a close relative; what documents to send |
| 6 | Sofia, Premier | my flight home was delayed 14 hours, what can I claim? | **nothing more** — $100 was already paid for this delay on 2 September |
| 7 | Priya | asthma attack in Porto, $2,300 hospital bill | covered — asthma was declared and accepted; $2,300 less $100 = **$2,200** |
| 8 | Sofia | backpack stolen on the beach while she swam | **no** — left unattended |

| File | What it is |
|---|---|
| `meridian.py` | the insurer: wording, limits, policies, claims, the tools, the questions and the answer key |
| `claims_chain.py` | the assistant in LangChain |
| `claims_scratch.py` | the assistant from this repo's parts |
| `claims_stubs.py` | stand-ins for every model and embedding call |
| `claims_checks.py` | what building it found out, each reproducible |
| `test_claims.py` | offline tests for both builds |

---
# Part 1 · Build

## What comes back

The answer is a `Reply` — fields a claims system can act on, not prose to be parsed. `payout` is only filled from a calculator result, never from the model's own arithmetic.

```python
# meridian.py
class Reply(BaseModel):
    """The assistant's answer, as fields a claims system can act on."""

    covered: Literal["yes", "no", "partly", "depends", "not_a_coverage_question"] = Field(
        description="Whether what the customer asks about is covered, according to the wording and their schedule.")
    answer: str = Field(description="The reply to the customer: plain, warm, specific to their policy.")
    payout: float | None = Field(
        default=None, description="The dollar amount they can expect, only if a calculator result gives it. "
                                  "Otherwise null.")
    clauses: list[str] = Field(description="The clause numbers the answer rests on, e.g. ['5.1', '5.2'].")
    next_steps: list[str] = Field(default_factory=list, description="What the customer has to do next, if anything.")
```

## Retrieval

The policy wording is split **one document per clause**, because the clause is the unit an answer cites. Chapter 4's Gemini `embed()` goes behind LangChain's `Embeddings` interface — two methods, one for documents and one for queries, which is also the split Gemini's task types make. Anything implementing them plugs into every LangChain vector store.

```python
# claims_chain.py
class GeminiEmbeddings(Embeddings):
    """Chapter 4's `embed()` behind LangChain's `Embeddings` interface.

    Two methods are the whole contract, and the split between them is the same one chapter 4
    makes with Gemini's task types: a document going into an index and a query searching it are
    embedded differently. Anything implementing these two plugs into every LangChain vector store.
    """

    def embed_documents(self, texts):
        from stores import embed
        return embed(texts, task="RETRIEVAL_DOCUMENT")

    def embed_query(self, text):
        from stores import embed
        return embed([text], task="RETRIEVAL_QUERY")[0]

def clause_documents():
    """One Document per clause: the clause is the unit a reply cites, so it is the unit retrieved."""
    return [Document(page_content=f"{number} {section}: {text}", metadata={"clause": number, "section": section})
            for number, (section, text) in M.WORDING.items()]

def build_retriever(embeddings=None, k=4):
    store = InMemoryVectorStore(embeddings or GeminiEmbeddings())
    store.add_documents(clause_documents())
    return store.as_retriever(search_kwargs={"k": k}).with_config(run_name="retrieve")
```

## Account facts: code, not the model's choice

The first version let the model pick its lookups in one tool-calling turn: *"call every tool you need at once"*. Measured over three questions, five times each, it called `lookup_policy` alone **15 times out of 15** — never the calculator, never the claims list (Part 2 re-runs this). A chain gives a model one turn, and this model uses one tool per turn.

So the lookups every answer needs — the schedule and the claims — are made by code, always. The model's only job is to **extract** what the conditional lookups need, through structured output; code calls the tools. LangChain tools are Runnables, so calling one is `tool.invoke(args)`, with the arguments checked against its schema.

```python
# meridian.py
class Lookups(BaseModel):
    """What the account lookups need from the customer's question."""

    claim_id: str = Field(default="", description="A claim number the customer mentions, e.g. CLM-20931. "
                                                  "Empty if none.")
    benefit: Literal["none", "baggage_item", "medical", "cancellation", "travel_delay"] = Field(
        description="The benefit a money question falls under; none if they are not asking about an amount "
                    "or a delay.")
    amount: float = Field(default=0.0, description="The dollar amount involved, exactly as stated. 0 if none.")
    hours: float = Field(default=0.0, description="For a travel delay, how many hours. 0 otherwise.")
```

```python
# claims_chain.py
def run_lookups(inputs):
    """The account facts, by code: the tools are Runnables, invoked with the fields the model extracted."""
    policy, asked = inputs["policy_number"], inputs["lookups"]
    calls = [("lookup_policy", {"policy_number": policy}), ("claims_for", {"policy_number": policy})]
    if asked.claim_id:
        calls.append(("claim_status", {"claim_id": asked.claim_id}))
    if asked.benefit != "none":
        calls.append(("estimate_payout", {"policy_number": policy, "benefit": asked.benefit,
                                          "amount": asked.amount, "hours": asked.hours}))
    return "\n".join(BY_NAME[name].invoke(args) for name, args in calls)
```

## The chain

`build_gather` is the first half. `RunnableBranch` skips the rewrite when there is no history. The two keys in one `RunnablePassthrough.assign(...)` — `docs` and `facts` — **run in parallel**: retrieval does not wait for the lookups. `with_fallbacks` means a failed extraction still produces an answer, just without a calculated figure.

```python
# claims_chain.py
def build_gather(model, retriever):
    """message + history -> question, the clauses it is about, and the account facts it needs."""
    rewrite = (ChatPromptTemplate.from_messages([("system", M.REWRITE_SYSTEM), MessagesPlaceholder("history"),
                                                 ("human", "{message}")])
               | model | StrOutputParser()).with_config(run_name="rewrite")
    # a first message has nothing to rewrite against, so it skips the model call
    standalone = RunnableBranch((lambda x: not x["history"], itemgetter("message")), rewrite)

    lookups = (ChatPromptTemplate.from_messages([("system", M.LOOKUPS_SYSTEM), ("human", "{question}")])
               | model.with_structured_output(M.Lookups, method="json_schema"))
    # if the extraction fails, the answer still gets the schedule and the claims -- just no figure
    lookups = lookups.with_fallbacks([RunnableLambda(lambda _: NOTHING_TO_CALCULATE)])
    # returns the extracted fields too, so `check` can apply the rules that need them
    look_up = (RunnablePassthrough.assign(lookups=lookups)
               | RunnablePassthrough.assign(facts=RunnableLambda(run_lookups))
               | RunnableLambda(lambda x: {"lookups": x["lookups"], "facts": x["facts"]})).with_config(run_name="look_up")

    return (RunnablePassthrough.assign(question=standalone)
            # two keys in one assign() run in parallel -- retrieval does not wait for the tools
            | RunnablePassthrough.assign(docs=itemgetter("question") | retriever, account=look_up)
            | RunnablePassthrough.assign(facts=lambda x: x["account"]["facts"], lookups=lambda x: x["account"]["lookups"],
                                         clauses=lambda x: format_clauses(x["docs"])))
```

Then respond, check, and the whole thing as one Runnable. `with_structured_output(..., method="json_schema")` on `ChatOpenAI` sends Groq a `response_format`, so the provider enforces the shape. When it cannot — the first live run got a 400 `output_parse_failed` with an empty `failed_generation` — the fallbacks run in the order chapter 1's `extract()` uses: JSON mode with pydantic checking the shape, then the second model. Each is a Runnable, so the order is one `with_fallbacks` call.

```python
# claims_chain.py
def build_respond(model, fallback=None):
    """clauses + facts -> a `Reply`, in the order chapter 1's `extract()` tries things.

    Schema mode first: Groq enforces the shape. When Groq cannot -- on the first live run it
    refused with `output_parse_failed` and an empty `failed_generation` -- JSON mode, with the
    shape in the prompt and pydantic checking it. Then the second model. Each is a Runnable, so
    the order is one `with_fallbacks` call.
    """
    tries = [_json_prompt() | model.with_structured_output(M.Reply, method="json_mode")]
    if fallback is not None:
        tries.append(ANSWER_PROMPT | fallback.with_structured_output(M.Reply, method="json_schema"))
    respond = (ANSWER_PROMPT | model.with_structured_output(M.Reply, method="json_schema")).with_fallbacks(tries)
    return respond.with_config(run_name="respond")

def check(turn):
    """The rules a reply must pass before anything acts on it. Code, not a prompt.

    Every clause it cites must have been in front of the model; and a payout must not repeat one
    already made (`meridian.enforce`).
    """
    retrieved = {doc.metadata["clause"] for doc in turn["docs"]}
    turn["unsupported"] = sorted(set(turn["reply"].clauses) - retrieved)
    turn["reply"], turn["held"] = M.enforce(turn["reply"], turn["policy_number"], turn.get("lookups"))
    return turn

def build_chain(model, retriever, fallback=None):
    """The whole assistant as one Runnable."""
    return (build_gather(model, retriever)
            | RunnablePassthrough.assign(reply=build_respond(model, fallback))
            | RunnableLambda(check).with_config(run_name="check"))
```

## A rule the model's own fields broke

`check` also runs `meridian.enforce`, added after the run in Part 2: both builds answered message 6 correctly in prose — Sofia's $100 for this delay was already paid — and with fields that would pay it again (`covered: yes, payout: 100.0`). Clause 3.2 says a delay is paid once, so that becomes code. Code cannot tell whether a second delay on one trip is the *same* delay, so it does not refuse: it blocks the payout and holds the reply for a person, with the reason.

```python
# meridian.py
def already_paid(policy_number, lookups):
    """Clause 3.2 as code: a travel delay already paid on this policy. The earlier claim, or None.

    Written after both builds' replies to message 6 said, in prose, that Sofia's delay was already
    paid -- and, in their fields, `covered: yes, payout: 100.0`. A system acting on the fields pays
    twice. Code cannot tell whether a second delay on the same trip is the SAME delay, so this does
    not refuse anything: it blocks the payout and holds the reply for a person.
    """
    if lookups is None or lookups.benefit != "travel_delay":
        return None
    for claim_id, claim in CLAIMS.items():
        if claim["policy"] == policy_number and claim["section"] == "travel delay" and claim.get("paid"):
            return f"{claim_id} already paid ${claim['paid']:.2f} for a travel delay on {claim['paid_on']}"
    return None

def enforce(reply, policy_number, lookups):
    """The reply as it may be acted on, and why it is held -- or the reply unchanged and None."""
    earlier = already_paid(policy_number, lookups)
    if earlier and reply.payout:
        return reply.model_copy(update={"payout": None}), f"payout blocked for review: {earlier}"
    return reply, None
```

## Memory, and why it is a list

`Assistant` holds the only state: each customer's messages so far, handed to the rewrite prompt's `MessagesPlaceholder`. A plain list, because LangChain 1.x has deprecated both of its own ways of keeping one — `RunnableWithMessageHistory` (*"Use LangGraph's built-in persistence instead"*) and, since 1.6.4, `InMemoryChatMessageHistory` itself. Conversation state is LangGraph's job now.

`batch` is worth noticing for what it is not: nothing in the chain implements it. Every composed Runnable has `invoke`, `batch` and `stream`.

```python
# claims_chain.py
class Assistant:
    """The chain, plus each customer's conversation so far. The only state anywhere is here.

    A conversation is a plain list of messages, handed to the prompt's `MessagesPlaceholder`.
    LangChain 1.x deprecated both of its own ways of keeping one -- `RunnableWithMessageHistory`
    ("use LangGraph's built-in persistence instead") and, in 1.6.4, `InMemoryChatMessageHistory`
    itself. Memory across turns is LangGraph's job now; what LangChain keeps is the message types.
    """

    def __init__(self, model=None, retriever=None, fallback=None):
        self.model = model or chat_model()
        self.retriever = retriever or build_retriever()
        self.gather = build_gather(self.model, self.retriever)
        self.respond = build_respond(self.model, fallback)
        self.chain = build_chain(self.model, self.retriever, fallback)
        self.streaming = build_streaming(self.model)
        self.histories = {}

    def _inputs(self, session, policy_number, message):
        return {"message": message, "policy_number": policy_number,
                "history": list(self.histories.get(session, []))}

    def _remember(self, session, message, answer):
        self.histories.setdefault(session, []).extend([HumanMessage(message), AIMessage(answer)])

    def ask(self, session, policy_number, message, callbacks=None):
        turn = self.chain.invoke(self._inputs(session, policy_number, message),
                                 config={"callbacks": callbacks or [], "run_name": f"turn:{session}"})
        self._remember(session, message, turn["reply"].answer)
        return turn

    def batch(self, requests, max_concurrency=4, callbacks=None):
        """Several customers at once. `batch` comes with every Runnable; nothing here implements it.

        `return_exceptions=True`: one customer's failure comes back as that customer's result
        instead of failing the other two -- the first live run lost a whole batch to one error.
        """
        turns = self.chain.batch([self._inputs(*r) for r in requests], return_exceptions=True,
                                 config={"max_concurrency": max_concurrency, "callbacks": callbacks or []})
        for (session, _, message), turn in zip(requests, turns):
            if not isinstance(turn, Exception):
                self._remember(session, message, turn["reply"].answer)
        return turns

    def stream(self, session, policy_number, message):
        """Yield the answer as it is written, then the finished reply with its fields checked.

        If the fields do not parse or validate, the enforced (non-streaming) call answers instead:
        the customer has already read the prose, so the fallback only has to get the fields right.
        """
        gathered = self.gather.invoke(self._inputs(session, policy_number, message))
        finished = ""
        for partial in self.streaming.stream(gathered):
            if "finished" in partial:
                finished = partial["finished"]
            else:
                yield partial
        try:
            reply = M.reply_from(finished)
        except ValueError:
            reply = self.respond.invoke(gathered)
        reply, _ = M.enforce(reply, policy_number, gathered.get("lookups"))
        self._remember(session, message, reply.answer)
        yield {"reply": reply}
```

## Streaming: the answer first, the fields after

`stream()` yields the answer to the customer as it is written. It cannot stream the structured reply, and the reason was measured, not assumed: **Groq delivers a JSON reply from gpt-oss in one piece** — with `response_format`, through LangChain and the raw OpenAI client alike (Part 2), and on this assistant's own prompt even without it: the first live run's "stream" arrived as a single 368-character partial.

Prose streams. So in one call the model writes the reply as plain text first, then a line `FIELDS:` and the fields as JSON. The stream shows the prose as it grows and never shows the marker; when the reply is finished, code splits it, parses the fields and validates them as a `Reply`. If they do not validate, the enforced call answers instead — the customer has already read the prose, so the fallback only has to get the fields right. The trade: on this path the *provider* no longer enforces the fields' shape; code checks it, the way chapter 1's `extract()` does when schema mode is unavailable.

The rule both builds share:

```python
# meridian.py
def visible_prose(text):
    """What of a reply still arriving is safe to show: everything before the marker, holding back a
    tail that might be the marker's first letters."""
    cut = text.find(FIELDS_MARKER)
    return text[:cut] if cut >= 0 else text[:max(0, len(text) - len(FIELDS_MARKER))]

def reply_from(text):
    """A finished prose-then-fields reply -> Reply. ValueError if the fields are missing or malformed."""
    import json
    prose, marker, rest = text.partition(FIELDS_MARKER)
    if not marker:
        raise ValueError("the reply has no FIELDS block")
    body = rest.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        fields = json.loads(body)
    except json.JSONDecodeError as bad:
        raise ValueError(f"FIELDS is not JSON: {bad}") from bad
    fields.pop("answer", None)
    return Reply(answer=prose.strip(), **fields)
```

`RunnableGenerator` turns a plain generator into a streaming step, so it composes with `|` like everything else:

```python
# claims_chain.py
def _prose_then_fields(chunks):
    """Streamed chunks -> the customer's answer so far, then the whole text once it is finished.

    A plain generator, made a streaming step by `RunnableGenerator`. It yields `{"answer": ...}`
    as the prose grows and stops showing text at the FIELDS marker, which the customer never sees.
    """
    text, shown = "", 0
    for chunk in chunks:
        text += chunk.content
        prose = M.visible_prose(text)
        if len(prose) > shown:
            shown = len(prose)
            yield {"answer": prose}
    yield {"answer": text.partition(M.FIELDS_MARKER)[0].rstrip(), "finished": text}

def build_streaming(model):
    """The respond step, streamed: prose first, then the fields as JSON after a marker.

    Why not stream the structured reply: Groq delivers a JSON reply from gpt-oss in one piece, with
    `response_format` (measured, LangChain and raw client alike: claims_checks.json_mode_streaming)
    and, on this assistant's prompt, without it -- the first live run's "stream" arrived as a single
    368-character partial. Prose streams. `Assistant.stream` checks the fields when it is done.
    """
    schema = json.dumps(M.fields_schema()).replace("{", "{{").replace("}", "}}")
    prompt = ChatPromptTemplate.from_messages([
        ("system", M.ANSWER_SYSTEM + M.PROSE_THEN_FIELDS.replace("{schema}", schema)), ("human", "{question}")])
    return prompt | model | RunnableGenerator(_prose_then_fields)
```

## Callbacks

`Steps` times every named step and counts tokens per model call, from events LangChain fires anyway. Nothing in the chain knows it exists: it is passed in `invoke`'s config and handed down to every Runnable inside — the same mechanism LangSmith uses to trace.

```python
# claims_chain.py
class Steps(BaseCallbackHandler):
    """Times every named step and counts the model's tokens, from LangChain's callback events.

    Nothing in the chain knows this exists. Callbacks are handed to `invoke` in its config and
    LangChain passes them down to every Runnable inside -- which is also how LangSmith traces.
    """

    NAMED = {"rewrite", "look_up", "retrieve", "respond", "check"}

    def __init__(self):
        self.rows, self._started = [], {}

    def _begin(self, name, run_id):
        if name in self.NAMED:
            self._started[run_id] = (name, time.perf_counter())

    def _end(self, run_id, detail=""):
        if run_id in self._started:
            name, began = self._started.pop(run_id)
            self.rows.append({"step": name, "seconds": round(time.perf_counter() - began, 2), "detail": detail})

    def on_chain_start(self, serialized, inputs, *, run_id, name=None, **kw):
        self._begin(name or kw.get("run_name"), run_id)

    def on_chain_end(self, outputs, *, run_id, **kw):
        self._end(run_id)

    def on_retriever_start(self, serialized, query, *, run_id, name=None, **kw):
        self._begin(name or "retrieve", run_id)

    def on_retriever_end(self, documents, *, run_id, **kw):
        self._end(run_id, "clauses " + ", ".join(d.metadata["clause"] for d in documents))

    def on_llm_end(self, response, **kw):
        usage = getattr(response.generations[0][0].message, "usage_metadata", None) or {}
        self.rows.append({"step": "  model call", "seconds": None,
                          "detail": f"{usage.get('input_tokens', 0)} in, {usage.get('output_tokens', 0)} out"})
```

## The same assistant, from scratch

The same four steps as plain functions called in order. Retrieval is chapter 4's `embed()` and `cosine()` over one vector per clause — which is all `InMemoryVectorStore` is — plus chapter 4's **floor**: a vector store always returns its k nearest, however far away they are, so a question the policy does not cover still gets four clauses to build an answer on. Below the floor, nothing is returned. The figure is measured on this wording, not chosen:

```python
# claims_scratch.py
# Chapter 4's floor: below this raw similarity a clause is not about the question at all. Measured on
# this wording with gemini-embedding-001, not chosen: every clause in the eight questions' top 4 scores
# 0.616 or more, and two questions the policy does not cover ("Can I bring my dog on the plane?", the
# head office's phone number) top out at 0.576. A thin margin from two examples; and the same figure
# chapter 4 measured on its journal. It cannot separate a needed clause from an unneeded one -- those
# overlap, 0.643-0.718 against up to 0.704 -- only "something relevant" from "nothing".
FLOOR = 0.6


class ClauseIndex:
    """One vector per clause, searched by cosine similarity. What `InMemoryVectorStore` is, written out --
    plus the one thing a vector store does not do: say it has nothing.

    A floor belongs to an embedding model and a corpus, so it applies by default only to the live one;
    a stand-in embedding scores on its own scale and passes its own floor, or none.
    """

    def __init__(self, embed=None, floor=None):
        if embed is None:
            from stores import embed
            floor = FLOOR if floor is None else floor
        self.embed, self.floor = embed, floor
        self.clauses = [(number, f"{number} {section}: {text}") for number, (section, text) in M.WORDING.items()]
        self.vectors = self.embed([text for _, text in self.clauses], task="RETRIEVAL_DOCUMENT")

    def search(self, query, k=4):
        from stores import cosine
        q = self.embed([query], task="RETRIEVAL_QUERY")[0]
        ranked = sorted(((cosine(q, v), clause) for v, clause in zip(self.vectors, self.clauses)), reverse=True)
        # the floor runs on RAW similarity, before anything else, as chapter 4's `Recall` does
        return [clause for score, clause in ranked[:k] if self.floor is None or score >= self.floor]
```

Extraction is chapter 1's `extract()` — schema mode, with validation and retry behind it; the lookups go through chapter 3's `Registry` and its argument check. Parallel is a thread pool, as chapter 5's `gather()` does it; memory is chapter 1's `Conversation`:

```python
# claims_scratch.py
def _gather(self, session, policy_number, message):
    from chat import Conversation
    conversation = self.conversations.setdefault(session, Conversation())
    history = list(conversation.messages)
    question = self._timed("rewrite", self.rewrite, history, message) if history else message
    with concurrent.futures.ThreadPoolExecutor() as pool:     # retrieval does not wait for the tools
        clauses = pool.submit(self._timed, "retrieve", self.index.search, question)
        asked = pool.submit(self._timed, "look_up", self.lookups, question)
        clauses, asked = clauses.result(), asked.result()
    self._asked = asked                   # for `enforce`, which needs what was extracted
    return conversation, question, clauses, run_lookups(policy_number, asked)

def ask(self, session, policy_number, message):
    conversation, question, clauses, facts = self._gather(session, policy_number, message)
    reply = self._timed("respond", self.respond, question, clauses, facts, policy_number)
    unsupported = sorted(set(reply.clauses) - {number for number, _ in clauses})
    reply, held = M.enforce(reply, policy_number, self._asked)
    conversation.messages += [{"role": "user", "content": message},
                              {"role": "assistant", "content": reply.answer}]
    return {"message": message, "question": question, "clauses": clauses, "facts": facts,
            "reply": reply, "unsupported": unsupported, "held": held}
```

Streaming is the same generator with no wrapper: the raw client's chunks go straight in.

```python
# claims_scratch.py
def prose_then_fields(chunks):
    """Text chunks -> the answer so far, then the whole text. What the chain's `RunnableGenerator` wraps."""
    text, shown = "", 0
    for delta in chunks:
        text += delta
        prose = M.visible_prose(text)
        if len(prose) > shown:
            shown = len(prose)
            yield {"answer": prose}
    yield {"answer": text.partition(M.FIELDS_MARKER)[0].rstrip(), "finished": text}
```

---
# Part 2 · Run and show

In [1]:
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.append(str(Path.cwd().parents[1] / "06_orchestration"))

from langchain_core.callbacks import get_usage_metadata_callback

import claims_chain as C
import claims_checks as K
import claims_scratch as X
import claims_stubs as S
import meridian as M
from meter import Meter                      # chapter 6's token meter

QUESTIONS = M.QUESTIONS

## 1 · Offline, both builds

Every model and embedding call replaced by `claims_stubs` — the same scripted "model" behind both builds, so any difference is wiring. Graded against the answer key:

In [2]:
offline = C.Assistant(model=S.Scripted(), retriever=C.build_retriever(S.WordEmbeddings()))
print("LangChain"); K.show_grades(K.grade([offline.ask(*q) for q in QUESTIONS]))
offline = X.Assistant(index=X.ClauseIndex(embed=S.embed), rewrite=S.rewrite, lookups=S.lookups, respond=S.respond)
print("\nfrom scratch"); K.show_grades(K.grade([offline.ask(*q) for q in QUESTIONS]))

LangChain
Q1 ok  yes                      payout=400.0    missing=[] unseen=['5.1']  My laptop was stolen from my hotel room in Lisbo
Q2 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  What's happening with my claim CLM-20931?
Q3 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  How long do I have to send it?
Q4 ok  no                       payout=None     missing=[] unseen=[]  I'm going skiing in Austria in December. If I br
Q5 ok  yes                      payout=None     missing=[] unseen=['1.1', '8.2']  My mum has just been taken into hospital and I m
Q6 ok  no                       payout=None     missing=[] unseen=['3.2']  My flight home from Tokyo was delayed 14 hours. 
Q7 ok  yes                      payout=2200.0   missing=[] unseen=['4.1']  Separately: I had a bad asthma attack in Porto a
Q8 ok  no                       payout=None     missing=[] unseen=['1.3']  Also, my backpack was stolen on the beach while 

8 of 8 right on coverage and p

Q1 ok  yes                      payout=400.0    missing=[] unseen=['5.1']  My laptop was stolen from my hotel room in Lisbo
Q2 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  What's happening with my claim CLM-20931?
Q3 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  How long do I have to send it?
Q4 ok  no                       payout=None     missing=[] unseen=[]  I'm going skiing in Austria in December. If I br
Q5 ok  yes                      payout=None     missing=[] unseen=['1.1', '8.2']  My mum has just been taken into hospital and I m
Q6 ok  no                       payout=None     missing=[] unseen=['3.2']  My flight home from Tokyo was delayed 14 hours. 
Q7 ok  yes                      payout=2200.0   missing=[] unseen=['4.1']  Separately: I had a bad asthma attack in Porto a
Q8 ok  no                       payout=None     missing=[] unseen=['1.3']  Also, my backpack was stolen on the beach while 

8 of 8 right on coverage and payout


8

The *"cited unseen"* column is the `check` step working: the stand-in embedding is crude, so it retrieves different clauses from the ones the scripted replies cite, and the check reports exactly which.

## 2 · LangChain, live

The three customers' first messages go through `batch` — concurrently, with nothing written to make it so. The rest go one at a time, in order, because they depend on the conversation before them.

In [3]:
assistant = C.Assistant(fallback=C.chat_model(C.FALLBACK))
first, rest = [0, 3, 5], [1, 2, 4, 6, 7]

with get_usage_metadata_callback() as usage:
    began = time.perf_counter()
    batched = assistant.batch([QUESTIONS[i] for i in first])
    print(f"3 customers' first messages, batched: {time.perf_counter() - began:.1f}s")
    for i, turn in zip(first, batched):
        if isinstance(turn, Exception):              # one customer's failure, not the batch's
            print(f"  message {i + 1} failed ({type(turn).__name__}: {str(turn)[:120]}); asking again")
    batched = [assistant.ask(*QUESTIONS[i]) if isinstance(t, Exception) else t for i, t in zip(first, batched)]
    steps = C.Steps()
    turns = dict(zip(first, batched))
    for i in rest:
        turns[i] = assistant.ask(*QUESTIONS[i], callbacks=[steps] if i == 2 else None)
chain_turns = [turns[i] for i in range(len(QUESTIONS))]
chain_tokens = sum(v["total_tokens"] for v in usage.usage_metadata.values())
print(f"tokens: {chain_tokens:,}\n")
chain_grades = K.grade(chain_turns)
K.show_grades(chain_grades)

3 customers' first messages, batched: 2.6s


tokens: 16,567

Q1 BAD depends                  payout=400.0    missing=[] unseen=[]  My laptop was stolen from my hotel room in Lisbo
Q2 BAD yes                      payout=None     missing=[] unseen=[]  What's happening with my claim CLM-20931?
Q3 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  How long do I have to send it?
Q4 ok  no                       payout=None     missing=[] unseen=[]  I'm going skiing in Austria in December. If I br
Q5 ok  yes                      payout=None     missing=[] unseen=[]  My mum has just been taken into hospital and I m
Q6 BAD yes                      payout=100.0    missing=[] unseen=[]  My flight home from Tokyo was delayed 14 hours. 
Q7 ok  yes                      payout=2200.0   missing=[] unseen=[]  Separately: I had a bad asthma attack in Porto a
Q8 ok  no                       payout=None     missing=[] unseen=[]  Also, my backpack was stolen on the beach while 

5 of 8 right on coverage and payout


5

The replies, as a customer would read them:

In [4]:
for i, turn in enumerate(chain_turns, 1):
    reply = turn["reply"]
    print(f"--- {i}. {turn['message']}")
    if turn["question"] != turn["message"]:
        print(f"    rewritten: {turn['question']}")
    print(f"    [{reply.covered}; payout {reply.payout}; clauses {', '.join(reply.clauses)}]")
    print("    " + reply.answer.replace("\n", "\n    "))
    print()

--- 1. My laptop was stolen from my hotel room in Lisbon. It was locked in the room safe, and it cost $1,800. How much will I get back?
    [depends; payout 400.0; clauses 5.1, 5.2, 1.3, 1.4]
    Your laptop is considered a valuable (clause 1.4) and was not unattended because it was locked in the hotel safe (definition in clause 1.3). Therefore the loss is potentially covered under clause 5.1. However, clause 5.2 requires a police report filed within 24 hours of the theft. Since we do not yet have that report, the claim is on hold. Once you provide a valid police report, the claim can be settled up to the single‑item limit of $500, less the $100 excess, which means a payout of $400.

--- 2. What's happening with my claim CLM-20931?
    [yes; payout None; clauses 8.1, 8.3]
    Hi Priya, your claim CLM-20931 for the stolen laptop is currently on hold because we are still waiting for the police report you were asked to provide on 2026-09-29. The claim was opened on 2026-09-26, which is wi

### What one turn looked like from the inside

Message 3, *"how long do I have to send it?"*, through the `Steps` callback — every named step, every model call, and which clauses retrieval brought back:

In [5]:
for row in steps.rows:
    print(f"{row['step']:14} {'' if row['seconds'] is None else str(row['seconds']) + 's':>7}  {row['detail']}")
print("\naccount facts it was given:\n" + chain_turns[2]["facts"])

  model call            438 in, 126 out
rewrite          0.65s  
retrieve          0.6s  clauses 8.3, 5.2, 8.2, 8.1
  model call            418 in, 93 out
look_up          0.66s  
  model call            748 in, 491 out
respond           3.6s  
respond           3.6s  
check             0.0s  

account facts it was given:
MT-48213: Priya Nair, Plus plan, Portugal 2026-09-10 to 2026-09-24. Add-ons: none. Accepted medical conditions: asthma.
CLM-20931: baggage, laptop stolen from hotel room, Lisbon, status on hold
CLM-20931: baggage claim, laptop stolen from hotel room, Lisbon, opened 2026-09-26, status: on hold. Waiting for: police report, requested on 2026-09-29.


### Streaming

A new message, streamed. Each line is the answer as the customer would see it so far; the last line is the reply, its fields parsed from after the marker and validated:

In [3]:
streamer = C.Assistant()
pieces = 0
for part in streamer.stream("daniel", "MT-51177", "If my bag doesn't arrive in Vienna, what does my plan pay?"):
    if "answer" in part:
        pieces += 1
        if pieces <= 6 or pieces % 10 == 0:
            print(f"{len(part['answer']):5} chars | ...{part['answer'][-60:]!r}")
    else:
        reply = part["reply"]
print(f"\n{pieces} updates to the answer while it was written")
print("fields:", {k: getattr(reply, k) for k in ("covered", "payout", "clauses", "next_steps")})

    1 chars | ...'Y'
    5 chars | ...'Your '
   14 chars | ...'Your bag not a'
   17 chars | ...'Your bag not arri'
   25 chars | ...'Your bag not arriving is '
   28 chars | ...'Your bag not arriving is tre'
   46 chars | ...'Your bag not arriving is treated as a loss of '
   71 chars | ...'t arriving is treated as a loss of baggage.\u202fUnder clause\u202f5.1'
  126 chars | ...'e\u202f5.1 we pay for belongings that are lost, stolen or damaged'
  169 chars | ...'stolen or damaged during your trip, up to the baggage limit '
  228 chars | ...' for the Essential plan. The exact amount you could receive '
  276 chars | ...'uld receive depends on the plan’s baggage limit and the exce'
  319 chars | ...'imit and the excess that applies to your policy.  \n\nTo work '
  365 chars | ...'y.  \n\nTo work out any payout you would need to tell us the v'
  406 chars | ...'ed to tell us the value of the items in the bag (or the tota'
  447 chars | ...'he bag (or the total claim amount) so we can ap

  490 chars | ...'n apply the plan’s limit and the $150 excess shown on your a'
  542 chars | ...'n your account. If the claim amount exceeds the excess, the '
  592 chars | ...'cess, the payable amount would be the lesser of the claim va'
  633 chars | ...'ser of the claim value and the plan’s baggage limit, minus t'
  670 chars | ...' baggage limit, minus the $150 excess.\n\n**Next steps**\n1. Pr'
  723 chars | ...'*\n1. Provide a detailed list of the contents of the missing '
  766 chars | ...'s of the missing bag and their values.  \n2. Submit a formal '
  827 chars | ...'aggage loss claim (including any airline documentation that '
  865 chars | ...'ne documentation that the bag was not delivered).  \n3. If an'
  916 chars | ...'\n3. If any items were stolen, include a police report filed '
  936 chars | ...'e stolen, include a police report filed within 24\u202fhours (cla'



215 updates to the answer while it was written
fields: {'covered': 'yes', 'payout': None, 'clauses': ['5.1'], 'next_steps': ['Provide a detailed list of the contents of the missing bag and their values', 'Submit a formal baggage loss claim with airline documentation that the bag was not delivered', 'If any items were stolen, include a police report filed within 24 hours']}


## 3 · From scratch, live

The same eight messages, in the same order, on the same model.

In [7]:
scratch = X.Assistant()
with Meter() as meter:
    scratch_turns = [scratch.ask(*q) for q in QUESTIONS]
scratch_cost = meter.summary()
print(scratch_cost, "\n")
scratch_grades = K.grade(scratch_turns)
K.show_grades(scratch_grades)

{'calls': 21, 'prompt': 11018, 'completion': 6075, 'total': 17093, 'seconds': 108} 

Q1 BAD depends                  payout=None     missing=[] unseen=[]  My laptop was stolen from my hotel room in Lisbo
Q2 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  What's happening with my claim CLM-20931?
Q3 ok  not_a_coverage_question  payout=None     missing=[] unseen=[]  How long do I have to send it?
Q4 ok  no                       payout=None     missing=[] unseen=[]  I'm going skiing in Austria in December. If I br
Q5 ok  yes                      payout=None     missing=[] unseen=[]  My mum has just been taken into hospital and I m
Q6 BAD yes                      payout=100.0    missing=[] unseen=[]  My flight home from Tokyo was delayed 14 hours. 
Q7 BAD yes                      payout=None     missing=[] unseen=[]  Separately: I had a bad asthma attack in Porto a
Q8 ok  no                       payout=None     missing=[] unseen=[]  Also, my backpack was stolen on the b

5

In [8]:
for i, turn in enumerate(scratch_turns, 1):
    reply = turn["reply"]
    print(f"--- {i}. [{reply.covered}; payout {reply.payout}; clauses {', '.join(reply.clauses)}]")
    print("    " + reply.answer[:400].replace("\n", "\n    "))

--- 1. [depends; payout None; clauses 5.2, 1.3, 1.4, 5.1]
    Your laptop is classified as a valuable (clause 1.4) and, because it was locked in the hotel room safe, it was not considered unattended (clause 1.3). Therefore the loss can be covered under the baggage clause (5.1) provided the required police report is supplied. Clause 5.2 requires a police report filed within 24 hours of the theft for any baggage theft claim. Please submit the police report (or
--- 2. [not_a_coverage_question; payout None; clauses 8.1, 8.3]
    Your claim CLM-20931 for the stolen laptop is currently on hold because we are waiting for the police report you were asked to provide. We requested this document on 2026-09-29, and under clause 8.3 you have 30 days from the request date to submit it. Once we receive the police report, we will continue processing your claim. Your claim was opened on 2026-09-26, which is within the 31‑day reporting
--- 3. [not_a_coverage_question; payout None; clauses 8.3]
    You h

## 4 · What building it found out

**One tool-calling turn.** The first design's lookup step, re-run: three questions, five times each, told to call every tool it needs at once.

In [9]:
print(K.as_json(K.one_shot_tools()))

{
 "Q1: My laptop was stolen from my hotel room in Lisbon. It was lo...": {
  "lookup_policy": 5
 },
 "Q6: My flight home from Tokyo was delayed 14 hours. What can I c...": {
  "lookup_policy": 5
 },
 "Q7: Separately: I had a bad asthma attack in Porto and spent a n...": {
  "lookup_policy": 5
 }
}


**Streaming structured output.** Content chunks per response mode, through LangChain and through the raw client:

In [10]:
print(K.as_json(K.json_mode_streaming()))

{
 "plain": {
  "LangChain content chunks": 54,
  "raw client content chunks": 45
 },
 "json_object": {
  "LangChain content chunks": 1,
  "raw client content chunks": 1
 },
 "json_schema": {
  "LangChain content chunks": 1,
  "raw client content chunks": 1
 }
}


**LangChain's own memory classes**, and **what `with_structured_output(method="json_schema")` sends**, depending on the model class:

In [11]:
print(K.as_json(K.deprecations()))
print(K.as_json(K.structured_output()))

{
 "InMemoryChatMessageHistory": "The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0",
 "RunnableWithMessageHistory": "RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead."
}
{
 "ChatOpenAI sends": [
  "response_format"
 ],
 "any other chat model sends": [
  "tool_choice",
  "tools"
 ],
 "and is told method=": "json_schema, which BaseChatModel.with_structured_output pops and ignores"
}


**A question the policy does not cover.** What each build hands the model, for one covered question and two the policy says nothing about (embeddings only, no model call):

In [2]:
print(K.as_json(K.out_of_scope()))

{
 "My laptop was stolen from my hotel room in Lisbon. It was lo": {
  "LangChain retriever (k=4)": [
   "5.2",
   "1.3",
   "1.4",
   "5.1"
  ],
  "scratch (k=4, floor 0.6)": [
   "5.2",
   "1.3",
   "1.4",
   "5.1"
  ]
 },
 "Can I bring my dog on the plane?": {
  "LangChain retriever (k=4)": [
   "1.1",
   "8.1",
   "9.1",
   "8.2"
  ],
  "scratch (k=4, floor 0.6)": []
 },
 "What's your head office's phone number?": {
  "LangChain retriever (k=4)": [
   "8.3",
   "1.1",
   "8.2",
   "8.1"
  ],
  "scratch (k=4, floor 0.6)": []
 }
}


## 5 · Side by side

In [4]:
import ast
import io
import tokenize


def code_lines(path):
    source = Path(path).read_text(encoding="utf-8")
    docstrings = set()
    for node in ast.walk(ast.parse(source)):
        body = getattr(node, "body", None)
        if isinstance(body, list) and body and isinstance(body[0], ast.Expr) \
                and isinstance(getattr(body[0], "value", None), ast.Constant) and isinstance(body[0].value.value, str):
            docstrings.update(range(body[0].lineno, body[0].end_lineno + 1))
    lines = set()
    for token in tokenize.generate_tokens(io.StringIO(source).readline):
        if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
                              tokenize.DEDENT, tokenize.ENDMARKER):
            lines.update(l for l in range(token.start[0], token.end[0] + 1) if l not in docstrings)
    return len(lines)


right = lambda grades: sum(g["covered ok"] and g["payout ok"] for g in grades)
unseen = lambda grades: sum(len(g["cited unseen"]) for g in grades)
print(f"{'':14}{'right':>7}{'cited unseen':>14}{'tokens':>9}{'lines of code':>15}")
print(f"{'LangChain':14}{right(chain_grades):>5}/8{unseen(chain_grades):>14}{chain_tokens:>9,}"
      f"{code_lines('claims_chain.py'):>15}")
print(f"{'from scratch':14}{right(scratch_grades):>5}/8{unseen(scratch_grades):>14}{scratch_cost['total']:>9,}"
      f"{code_lines('claims_scratch.py'):>15}")

                right  cited unseen   tokens  lines of code
LangChain         5/8             0   16,567            165
from scratch      5/8             0   17,093            125


## 6 · Message 6, with the rule

Parts 2 and 3 ran before `enforce` existed. Message 6 again, through both builds, with it — Sofia's first message, so it needs nothing before it:

In [2]:
for name, build in [("LangChain", C.Assistant()), ("from scratch", X.Assistant())]:
    turn = build.ask(*QUESTIONS[5])
    reply = turn["reply"]
    print(f"--- {name}: [{reply.covered}; payout {reply.payout}; clauses {', '.join(reply.clauses)}]")
    print(f"    held: {turn['held']}")
    print("    " + reply.answer[:300].replace("\n", " "))

--- LangChain: [yes; payout None; clauses 3.1, 3.2]
    held: payout blocked for review: CLM-20877 already paid $100.00 for a travel delay on 2026-09-02
    Your return flight from Tokyo was delayed by 14 hours, which qualifies for the travel‑delay benefit under your Premier plan. The policy (clause 3.1) pays $50 for each full 6‑hour block of delay, up to the plan maximum, and clause 3.2 states the benefit is paid only once for the same delay. Two full 


--- from scratch: [yes; payout None; clauses 3.1, 3.2]
    held: None
    Your travel delay claim is covered under clause 3.1 of your Premier plan. A 14‑hour delay counts as two full 6‑hour periods, giving you a benefit of $50 × 2 = $100, which has already been paid under claim CLM‑20877 on 2026‑09‑02. No further amount is payable.


## Findings

**Both builds scored 5 of 8 on the strict check, at the same cost** — 16,567 tokens against 17,093, one run each. Same prompts, same model: the framework does not change what the model is asked, so it does not change what the model gets wrong.

**Read rather than counted, the misses are not all equal.**
- Message 1, *"depends"* on the police report (both builds), and message 2, a status question labelled *"yes"* (LangChain): defensible readings of the wording and of a schema field, not wrong answers.
- Message 6 is the real one, and **both builds made it**: the prose correctly says the $100 for Sofia's delay was already paid, while the fields say `covered: yes, payout: 100.0`. A claims system acting on the fields would pay her twice. The prose and the fields disagree inside one reply. The fix went where `check` already was — in code, not in the prompt: `enforce` blocks a payout that repeats one already made and holds the reply for a person (section 6). Re-run once with it, LangChain's reply again said `payout: 100.0` and was held with the earlier claim named; the scratch build's reply left the payout empty this time, so there was nothing to block. Either way, nothing acting on the fields pays twice.
- Message 7: the scratch build gave no figure; LangChain's said $2,200. One run each, so not something to pin on either.

**The LangChain build is longer**: 165 lines of code against 125. The scratch build stands on chapters 1, 3 and 4, and a pipeline this linear is short in plain Python. What the extra lines buy is in the next table.

| | LangChain gave us | From scratch |
|---|---|---|
| composing steps | `\|`, `RunnablePassthrough.assign` — two keys in one `assign` run in parallel | calls in order, a thread pool for the parallel part |
| `invoke` / `batch` / `stream` | on every composed chain, for free; `batch(return_exceptions=True)` isolates failures | written per method (`batch` is a pool.map) |
| fallbacks and retries | `with_fallbacks([...])` around any step, in order | try/except, by hand, in each place |
| observability | callbacks passed in `invoke`'s config reach every step — the hook LangSmith uses | a timer per step, and chapter 6's `Meter` |
| retrieval | `Embeddings`, `Document`, a vector store, a retriever — Gemini plugged in with two methods; always the k nearest | `embed()`, `cosine()`, a sort, and chapter 4's floor: nothing for a question the policy does not cover |
| tools | schema'd Runnables: `tool.invoke(args)` validates | chapter 3's `Registry` |
| conversation memory | **nothing current** — both classes deprecated in favour of LangGraph | chapter 1's `Conversation` |

**What building it found out**, each in Part 2:
1. **A chain gives the model one turn, and this model uses one tool per turn.** Told to call every tool it needed at once, it called `lookup_policy` alone in **15 of 15** trials. Lookups every answer needs became code; the model only extracts what the conditional ones need.
2. **Structured output on Groq does not stream.** JSON-mode replies arrive as one chunk, with LangChain and the raw client alike, and a prompted JSON reply arrived whole as well. Prose streams: the reply is written prose first and fields after a marker, which streamed in 215 updates and still produced checked fields.
3. **`JsonOutputParser` yields nothing while any text precedes the JSON** — `'Sure.\n{...'` parses to `None` until a closing code fence arrives.
4. **`with_structured_output(method="json_schema")` means different things on different model classes.** On `ChatOpenAI` it is the provider's JSON schema; on any other class the base implementation pops `method` without a word and forces a tool call.
5. **Schema mode can fail with nothing to show for it.** Groq's 400 `output_parse_failed` came with an empty `failed_generation`. The fallback order chapter 1's `extract()` already used — schema, then JSON mode with validation, then another model — is one `with_fallbacks` call here.
6. **LangChain 1.x has handed memory to LangGraph.** `RunnableWithMessageHistory` and `InMemoryChatMessageHistory` are both deprecated. What LangChain keeps is the building blocks; what happens across turns is the next notebook's framework.

**Where LangChain earns its place** is the uniform interface. Every step and every composition of steps is the same kind of object, so `batch`, fallbacks and callbacks work on all of them without the steps knowing — and the callback hook is how LangSmith will trace this assistant in notebook 7. **What it costs** is that the uniform surface hides differences underneath: finding 4 is the same method call doing two different things, and findings 2 and 3 are behaviours you only meet by running against a real provider.